In [ ]:
import MeshFEM
import mesh_energy, panelization, py_newton_optimizer, benchmark, sim_utils
import mesh, parallelism, fd_validation, energy, viewer, loads
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh', embeddingDimension=3)

optVars = mesh_energy.NodalVars(m, 3) # Create per-node position variables (the variable dimension 3 here can also be inferred).
em = MeshFEM.EmbeddedMesh(m, optVars) # A wrapper object used to visualize the deformation described by `optVars`.

import closest_point_projection
proj = closest_point_projection.ClosestPointProjection(m)
springs = loads.ProjectedSprings(optVars, np.arange(m.numVertices(), dtype=np.int32), proj, 1e3)

prob = py_newton_optimizer.NewtonMultiobjectiveProblem(optVars, [springs])

In [ ]:
# Keep the mesh boundary vertices on the ground (but alllow them to slide in plane)
prob.setFixedVars(sim_utils.getBBoxVars(em, sim_utils.BBoxFace.MIN_Z, tol=1e-2, displacementComponents=[2]))

In [ ]:
import pickle, lzma
data = pickle.load(lzma.open('data/bad.pkl.gz', 'r'))

In [ ]:
prob.setVars(data['x'])

In [ ]:
fd_validation.gradConvergencePlot(prob, perturb=data['perturb'])

In [ ]:
perturb_nofixed = data['perturb'].copy()
perturb_nofixed[prob.fixedVars()] = 0

In [ ]:
fd_validation.gradConvergencePlot(prob, perturb=perturb_nofixed)

In [ ]:
benchmark.reset()
fd_grad = fd_validation.fdGrad(prob, 1e-7)
benchmark.report()

In [ ]:
diff = (fd_grad - prob.gradient()).reshape(-1, 3)
diffNorms = np.linalg.norm(diff, axis=1)
order = np.argsort(diffNorms)

In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.hist(diffNorms, bins=50)
plt.yscale('log')
plt.subplot(1, 2, 2)
plt.loglog(np.linspace(0, 1, len(diff)), diffNorms[order])

In [ ]:
vdiff = viewer.Viewer(em, scalarField=diffNorms)
vdiff.show()

In [ ]:
ns = springs.numSprings()
P = np.array([(springs.attachmentPointB(s).position, springs.attachmentPointB(s).preprojectedPosition) for s in range(ns)]).reshape(-1, 3)
E = np.column_stack((2 * np.arange(ns), 2 * np.arange(ns) + 1)) 
lv = viewer.Viewer((P, E), superView=vdiff)
lv.showPoints()